In [48]:
import pandas as pd

In [49]:
netflix_tv_show_df = pd.read_csv("./titles.csv")
netflix_tv_show_df = netflix_tv_show_df.dropna(axis=0)
netflix_tv_show_df = netflix_tv_show_df.rename({"tmdb_popularity": "imdb_popularity"}, axis=1)
netflix_tv_show_df.drop(["id", "description", "production_countries", "imdb_id", "tmdb_score","age_certification","runtime","seasons","imdb_votes","release_year"], axis=1, inplace=True)
netflix_tv_show_df.head()

,title,type,genres,imdb_score,imdb_popularity
5,Monty Python's Flying Circus,SHOW,"['comedy', 'european']",8.8,12.919
29,Monty Python's Fliegender Zirkus,SHOW,['comedy'],8.1,1.490
47,Seinfeld,SHOW,['comedy'],8.9,128.743
55,Knight Rider,SHOW,"['action', 'scifi', 'crime', 'drama']",6.9,44.378
57,Thomas & Friends,SHOW,"['family', 'comedy', 'music', 'action', 'anima...",6.5,49.384


In [50]:
netflix_tv_show_df = netflix_tv_show_df[netflix_tv_show_df.type != "MOVIE"]
netflix_tv_show_df.drop("type", axis=1, inplace=True)
netflix_tv_show_df.head()

,title,genres,imdb_score,imdb_popularity
5,Monty Python's Flying Circus,"['comedy', 'european']",8.8,12.919
29,Monty Python's Fliegender Zirkus,['comedy'],8.1,1.490
47,Seinfeld,['comedy'],8.9,128.743
55,Knight Rider,"['action', 'scifi', 'crime', 'drama']",6.9,44.378
57,Thomas & Friends,"['family', 'comedy', 'music', 'action', 'anima...",6.5,49.384


In [51]:
netflix_tv_show_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1597 entries, 5 to 5796
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   title            1597 non-null   object 
 1   genres           1597 non-null   object 
 2   imdb_score       1597 non-null   float64
 3   imdb_popularity  1597 non-null   float64
dtypes: float64(2), object(2)
memory usage: 62.4+ KB


In [52]:
import ast
netflix_tv_show_df['genres'] = netflix_tv_show_df['genres'].apply(ast.literal_eval)

In [53]:
# One Hot Encode And Flatten the Genres
genre_df = netflix_tv_show_df['genres'].apply(lambda x: pd.Series({genre:1 for genre in x}))
genre_df = genre_df.fillna(0).astype(int)
netflix_tv_show_df = pd.concat([netflix_tv_show_df,genre_df], axis =1)


In [38]:
netflix_tv_show_df.drop(columns=['genres'], axis=1, inplace=True)

In [54]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
netflix_tv_show_df['imdb_popularity_scaled'] = scaler.fit_transform(netflix_tv_show_df[['imdb_popularity']])

In [46]:
netflix_tv_show_df.drop(columns=['imdb_popularity'], axis=1, inplace=True)

In [ ]:
netflix_tv_show_df[['imdb_popularity_scaled']]

In [62]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=15, random_state=42)

columns_as_features = [column_name for column_name in netflix_tv_show_df.columns if column_name not in ['title','imdb_popularity','genres']]

netflix_tv_show_df['clusters'] = kmeans.fit_predict(netflix_tv_show_df[columns_as_features])

In [63]:
netflix_tv_show_df[['title','imdb_popularity','genres','clusters']].head(100)

,title,imdb_popularity,genres,clusters
5,Monty Python's Flying Circus,12.919,"[comedy, european]",1
29,Monty Python's Fliegender Zirkus,1.490,[comedy],1
47,Seinfeld,128.743,[comedy],0
55,Knight Rider,44.378,"[action, scifi, crime, drama]",14
57,Thomas & Friends,49.384,"[family, comedy, music, action, animation, fan...",4
...,...,...,...,...
427,Ben & Holly's Little Kingdom,18.311,"[comedy, family, fantasy, animation, european]",4
428,The Garfield Show,19.716,"[animation, comedy, family, european]",9
430,Wakfu,48.590,"[action, drama, scifi, fantasy, animation, fam...",11
439,Vampire Knight,16.465,"[scifi, romance, animation, action, drama, fan...",12


In [64]:
netflix_tv_show_df.loc[netflix_tv_show_df['title'] == 'Squid Game']

,title,genres,imdb_score,imdb_popularity,comedy,european,action,scifi,crime,drama,...,reality,western,thriller,horror,documentation,sport,war,history,imdb_popularity_scaled,clusters
4863,Squid Game,"[action, drama, thriller]",8.0,272.707,0,0,1,0,0,1,...,0,0,1,0,0,0,0,0,2.572397,13


In [67]:
netflix_tv_show_df.loc[netflix_tv_show_df['clusters'] == 13].head(8)

,title,genres,imdb_score,imdb_popularity,comedy,european,action,scifi,crime,drama,...,reality,western,thriller,horror,documentation,sport,war,history,imdb_popularity_scaled,clusters
243,Breaking Bad,"[drama, thriller, crime]",9.5,337.419,0,0,0,0,1,1,...,0,0,1,0,0,0,0,0,3.270037,13
249,Criminal Minds,"[thriller, crime, drama]",8.1,401.416,0,0,0,0,1,1,...,0,0,1,0,0,0,0,0,3.959969,13
250,NCIS,"[action, drama, crime, thriller, comedy]",7.8,254.207,1,0,1,0,1,1,...,0,0,1,0,0,0,0,0,2.372954,13
253,Supernatural,"[scifi, thriller, drama, fantasy, horror]",8.5,406.666,0,0,0,1,0,1,...,0,0,1,1,0,0,0,0,4.016568,13
272,The Vampire Diaries,"[scifi, drama, thriller, fantasy, horror, roma...",7.7,463.661,0,0,0,1,0,1,...,0,0,1,1,0,0,0,0,4.631013,13
455,The Cartel,"[action, crime, drama, thriller, comedy]",8.1,208.108,1,0,1,0,1,1,...,0,0,1,0,0,0,0,0,1.875975,13
783,Pablo Escobar: The Drug Lord,"[crime, drama, history, thriller]",8.4,314.603,0,0,0,0,1,1,...,0,0,1,0,0,0,0,1,3.024065,13
915,Outlander,"[scifi, drama, fantasy, romance]",8.4,238.679,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,2.205551,13
